### What's so far? 

Following topics are re-implemented from the [article](https://towardsdatascience.com/all-you-need-to-know-about-vector-databases-and-how-to-use-them-to-augment-your-llm-apps-596f39adfedb/#003d) in `vector-embedding-store/all-you-need-to-know-vector-db/implementation.ipynb` notebook. 

**Vector Embedding generation for texts**
- Took a simple simple list of strings as `text_chunk`, which is of our interest to get vector embedding and store the vectors in a vector store.
- A function called `_get_embeddings` that takes the `text_chunk` as input, defines a Huggingface InferenceClient. The client has a method called `feature_extraction` that takes `text_chunk` and the embedding model's id as input and gives embedding as output. `_get_embeddings` function returns the same.
- A function called `from_text_to_embeddings` that takes `text_chunk` as input, creates a pandas dataframe dataframe out of it with `text_chunk` as column, it creates another column `embeddings` and populates it with corresponding text embedding by applying the `_get_embeddings` function over the `text_chunk` column. Then it splits the embeddings column into individuell columns for each vector dimension and returns the dataframe.

**Plotting 384D vector in 2D using PCA**
- Plotting of the 384 dimentional vector embedding to a 2D plot by applying PCA.

**Calculate the distance and similarity score**
- A function called `calculate_cosine_similarity` that takes a new query text and the existing dataframe as input. Calculates the cosine similarity between the embeddings of the new query text and every other entry in the dataframe. Returns a dataframe with the cosine similarity score.

**Store vector embeddings in a vector DB**
- Then created a chromadb PersistentClient and created a vectorstore (collection).
- In the dataframe last column was text_chunk or corresponding sentences. Combined all dimensions of the vector embeddings to one array, converted that into list, also list of text_chunk into list, and ids: these were added to the vectorstore.
- Also a function to get all the entries in the vectorstore.
- And finally a function `query_vector_database` to query the vector db and get top `n` similar texts or documents.
- Tested with few examples.

There are lots of areas need clarity and clarification of doubts, that's why the following experiments:

In [25]:
import numpy as np
import pandas as pd
import chromadb
from chromadb.config import Settings

### Different ways of getting embedding vector

### Tokenization VS Vector Embedding VS Chunking

### Explore what's inside ChromaDB Vectorstore

The directory structure inside the selected folder for vectorstore:
```
ChromaDB_Vectorestore/
├── chroma.sqlite3
└── a_long_hash_id/
    ├── data_level0.bin
    ├── header.bin
    ├── length.bin
    └── link_lists.bin
```

First I'm going to explore what's inside `chroma.sqlite3` DB file. 

In [5]:
import sqlite3
from pprint import pprint

In [11]:
conn = sqlite3.connect("ChromaDB_Vectorstore/chroma.sqlite3")
cursor = conn.cursor()  # CONNECTION OPENED, CLOSE IT ONCE DONE..!

# List tables inside sqlite3 metadata db..
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
pprint(cursor.fetchall())

[('migrations',),
 ('acquire_write',),
 ('collection_metadata',),
 ('segment_metadata',),
 ('tenants',),
 ('databases',),
 ('collections',),
 ('maintenance_log',),
 ('segments',),
 ('embeddings',),
 ('embedding_metadata',),
 ('max_seq_id',),
 ('embedding_fulltext_search',),
 ('embedding_fulltext_search_data',),
 ('embedding_fulltext_search_idx',),
 ('embedding_fulltext_search_content',),
 ('embedding_fulltext_search_docsize',),
 ('embedding_fulltext_search_config',),
 ('embeddings_queue',),
 ('embeddings_queue_config',)]


**Comments:**
- `chroma.sqlite3` is the metadata database. This is a normal SQLite file that stores structured info about the vectors, not the vectors themselves.

In [19]:
# Query tables
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
for (table_name,) in tables:
    print(f"\n--- {table_name} ---")
    # try:
    rows = cursor.execute(f"SELECT * FROM {table_name} LIMIT 3;").fetchall()
    for row in rows:
        print(row)
        print()
    # except Exception as e:
    #     print(f"Error reading {table_name}: {e}")


--- migrations ---
('sysdb', 1, '00001-collections.sqlite.sql', 'CREATE TABLE collections (\n    id TEXT PRIMARY KEY,\n    name TEXT NOT NULL,\n    topic TEXT NOT NULL,\n    UNIQUE (name)\n);\n\nCREATE TABLE collection_metadata (\n    collection_id TEXT REFERENCES collections(id) ON DELETE CASCADE,\n    key TEXT NOT NULL,\n    str_value TEXT,\n    int_value INTEGER,\n    float_value REAL,\n    PRIMARY KEY (collection_id, key)\n);\n', '38352d725ad1c16074fac420b22b4633')

('sysdb', 2, '00002-segments.sqlite.sql', 'CREATE TABLE segments (\n    id TEXT PRIMARY KEY,\n    type TEXT NOT NULL,\n    scope TEXT NOT NULL,\n    topic TEXT,\n    collection TEXT REFERENCES collection(id)\n);\n\nCREATE TABLE segment_metadata (\n    segment_id TEXT  REFERENCES segments(id) ON DELETE CASCADE,\n    key TEXT NOT NULL,\n    str_value TEXT,\n    int_value INTEGER,\n    float_value REAL,\n    PRIMARY KEY (segment_id, key)\n);\n', '2913cb6a503055a95f625448037e8912')

('sysdb', 3, '00003-collection-dimension

In [13]:
# collections, segments, embedddings, embedding_metadata, max_seq_id, embedding_fulltext_search, embedding_fulltext_search_data
def print_table(table_name):
    rows = cursor.execute(f"SELECT * FROM {table_name} LIMIT 3;").fetchall()
    for row in rows:
        pprint(row)
        print()

**Notes:**
- There are different types of tables. 
- Core Tables:
    - collections - logical groups of embeddings (like “documents”, “chat history”)
    - embeddings - references to stored vectors (IDs, not raw vectors)
    - embedding_metadata - key-value metadata (e.g., source, text, tags)
- System tables
    - segments - chunks of vector index data
    - tenants, databases - multi-tenant support
    - migrations - schema versioning
- Search-related tables
    - embedding_fulltext_search* - full-text index (for keyword search alongside vectors)

In [14]:
print_table('collections')

('86bd4b2e-5b0c-49d9-9a3c-91aa6291e405',
 'my_collection',
 384,
 '00000000-0000-0000-0000-000000000000',
 '{}',
 '{"defaults":{"string":{"fts_index":{"enabled":false,"config":{}},"string_inverted_index":{"enabled":true,"config":{}}},"float_list":{"vector_index":{"enabled":false,"config":{"space":"l2","embedding_function":{"type":"known","name":"default","config":{}},"hnsw":{"ef_construction":100,"max_neighbors":16,"ef_search":100,"num_threads":16,"batch_size":100,"sync_threshold":1000,"resize_factor":1.2}}}},"sparse_vector":{"sparse_vector_index":{"enabled":false,"config":{"embedding_function":{"type":"unknown"},"bm25":false}}},"int":{"int_inverted_index":{"enabled":true,"config":{}}},"float":{"float_inverted_index":{"enabled":true,"config":{}}},"bool":{"bool_inverted_index":{"enabled":true,"config":{}}}},"keys":{"#document":{"string":{"fts_index":{"enabled":true,"config":{}},"string_inverted_index":{"enabled":false,"config":{}}}},"#embedding":{"float_list":{"vector_index":{"enabled":

**Comments**
- First it shows a hash `86bd4b2e-5b0c-49d9-9a3c-91aa6291e405` which is not same with the folder, this hash is for metadata
- It also saves name of collection or vectorstore db name, here `my_collection`
- Then, embedding dimention.

**This `collections` table is: logical groups of embeddings (like “documents”, “chat history”)**

In [15]:
print_table('embeddings')

(1, '85dfec62-7c1a-40fa-a5d3-7ff02d9f8a82', '0', 1, '2026-03-27 13:34:04')

(2, '85dfec62-7c1a-40fa-a5d3-7ff02d9f8a82', '1', 2, '2026-03-27 13:34:04')

(3, '85dfec62-7c1a-40fa-a5d3-7ff02d9f8a82', '2', 3, '2026-03-27 13:34:04')



**`embeddings` table: references to stored vectors (ie, IDs, not raw vectors)**

In [16]:
print_table('embedding_metadata')

(1, 'chroma:document', 'The sky is blue.', None, None, None)

(2, 'chroma:document', 'The grass is green.', None, None, None)

(3, 'chroma:document', 'The sun is shining.', None, None, None)



**`embedding_metadata` table: key-value metadata (e.g., source, text, tags)**

In [18]:
print_table('segments')

('a146bfb5-def2-4e54-bf5c-9aa322d45a1d',
 'urn:chroma:segment/vector/hnsw-local-persisted',
 'VECTOR',
 '86bd4b2e-5b0c-49d9-9a3c-91aa6291e405')

('85dfec62-7c1a-40fa-a5d3-7ff02d9f8a82',
 'urn:chroma:segment/metadata/sqlite',
 'METADATA',
 '86bd4b2e-5b0c-49d9-9a3c-91aa6291e405')



**`segments` table: chunks of vector index data**

In [26]:
VECTOR_STORE_PATH = 'ChromaDB_Vectorstore/'
COLLECTION_NAME = 'my_collection'
embeddings_df = pd.read_csv('saved_artifacts/embeddings_df.csv')
def get_or_create_client_and_collection(VECTOR_STORE_PATH, COLLECTION_NAME):
    # get/create a chroma client
    chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
    # get or create collection
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
    return collection
# get or create collection
collection = get_or_create_client_and_collection(VECTOR_STORE_PATH, COLLECTION_NAME)
def query_vector_database(VECTOR_STORE_PATH, COLLECTION_NAME, query, n=2):
    # query collection
    results = collection.query(
        query_texts=query,
        n_results=n
    )
    print(f"Similarity Search: {n} most similar entries:")
    print(results["documents"])
    return results
similar_vector_entries = query_vector_database(VECTOR_STORE_PATH, COLLECTION_NAME, query=["The earth orbits the sun."])
similar_vector_entries

Similarity Search: 2 most similar entries:
[['The moon orbits the Earth.', 'The sun is shining.']]


{'ids': [['9', '2']],
 'embeddings': None,
 'documents': [['The moon orbits the Earth.', 'The sun is shining.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.5676224827766418, 0.8526027798652649]]}

In [28]:
collection.get()

{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
 'embeddings': None,
 'documents': ['The sky is blue.',
  'The grass is green.',
  'The sun is shining.',
  'I love chocolate.',
  'Pizza is delicious.',
  'Coding is fun.',
  'Roses are red.',
  'Violets are blue.',
  'Water is essential for life.',
  'The moon orbits the Earth.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None, None, None, None, None, None, None, None, None, None]}

**A BIG DOUBT**
- When I print the `embedding_metadata` table from `chroma.sqlite3` db, It is showing only 3 sentences saved inside the metadata sqlite db, but querying it is showing some other sentence (although it is showing correct and relavant sentance). But why it is not showing all the sentence in the `embedding_metadata`?
- I saved metadata as well as the text_chunks also, and there are 10 sentences or text_chunks, but where are those saved in vectorstore?
- How by executing `collection.get()` gives all the 10 sentences?

**OKAY, LET'S MOVE FORWARD TOWARDS THE FOLDER WITH A_LONG_HASH_ID:**

- Vector index (the real embeddings)
- This hashed folder is where the actual vector index lives.
- Chroma uses an approximate nearest neighbor index (typically Hierarchical Navigable Small World graph / HNSW).

Each file has a specific role:
- data_level0.bin
    - Stores the actual embedding vectors
    - Binary format (float arrays)
    - This is the numerical data for similarity search
- header.bin
    - Metadata about the index:
        - vector dimension (e.g., 384, 768, 1536)
        - number of elements
        - configuration parameters
- length.bin
    - Stores sizes / offsets
    - Helps locate vectors efficiently inside binary files
- link_lists.bin
    - The graph structure of the HNSW index
    - Contains neighbor connections between vectors
    - Enables fast approximate nearest neighbor search

**How it all works together**

When we query Chroma:
1. SQLite (chroma.sqlite3)
    - Finds which collection / embeddings to search
    - Applies filters (metadata, etc.)
2. Binary index (a_long_hash_id/)
    - Loads vectors
    - Uses HNSW graph (link_lists.bin) to find nearest neighbors quickly